# Modelos finales v2 · árboles de Willy y clásicos de Samuel · TFM Energía UCM

Amplía `06_modelos_finales.ipynb` con las familias que el equipo entrenó fuera de su arnés y **lo rehace todo sin Trayport**. El entrenamiento vive en `scripts/entrenar_finales_v2.py`, que importa las funciones de `entrenar_finales.py` en vez de copiarlas; este notebook lo lanza, si se quiere, y presenta los resultados.

| familia | autor | tipo | entrada | objetivo | semillas |
|---|---|---|---|---|---|
| `seq2seq`, `gru`, `simplernn`, `lstm`, `conv1d_lstm`, `seq2seq_absoluto`, `denso` | Torgio | red | tensores: 168 h, decoder de 24 y estáticos | residuo frente al precio de D (salvo `seq2seq_absoluto`) | 3 |
| `boosting` | Torgio | árbol | vista plana del encoder, 24 modelos | residuo | 3 |
| `lgbm_nucleo` | Willy | árbol | matriz plana, una fila por hora, un solo modelo | precio | 3 |
| `ridge`, `elasticnet` | Samuel | lineal | matriz plana + filtro de Spearman | precio | 1 |
| `sarima`, `sarimax` | Samuel | estadístico | serie horaria del precio (+ exógenas) | precio | 1 |

Los lineales y los SARIMA son deterministas: tres semillas darían tres veces el mismo modelo y una desviación de 0,000 que no mide nada.

**Lo que corrige respecto a 06**: mide la copia de la curva del día D en cada modelo y documenta el ancla semanal que se probó para corregirla y se descartó (sección 3); respeta la frontera de producción, sin los retardos meteorológicos de reanálisis que a las 11:00 no existen (sección 4); tipifica las exógenas del SARIMAX y le añade el término constante; y elige el modelo final con una selección de ensemble decidida solo con validación (sección 7).

**Lo que no cambia**: la matriz, el corte (entrenamiento hasta 2024, validación 2025, test enero–julio 2026), la función `metricas` y los días evaluados. El test no interviene en ninguna decisión.

## Aviso · sin Trayport

`co2_eua_dec` y `gas_ttf_m1` eran vistas sobre `trayport_daily_ohlc`, la fuente que el equipo retiró (nota 54 de `docs/notas_memoria_tfm.md`). El script las quita de la matriz **antes** de construir cualquier entrada —tensores y matriz plana— y para en seco si alguna lista de features las contiene.

Los resultados de `data/gold/finales_nucleo/` (30-ago) se entrenaron **con** ellas, como dicen sus `.preprocesado.json`, así que aquí no se reutilizan: todo sale en `data/gold/finales_v2_<matriz>/`, sin tocar la carpeta anterior.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "scripts"))
import entrenar_finales_v2 as V2

MATRIZ = "nucleo"
PRUEBA = os.environ.get("TFM_V2_PRUEBA") == "1"   # lee la carpeta *_prueba, para comprobar el notebook
ANCLA = V2.ANCLA_DEFECTO     # "D"; "semanal" lee el entrenamiento con el ancla descartada
VETOS = V2.VETOS_DEFECTO     # ("met_lags",); con "ree_prev" además, la ablación de las previsiones REE
SALIDA = V2.carpeta_salida(MATRIZ, PRUEBA, ANCLA, VETOS)
print("salida:", SALIDA.relative_to(REPO))

In [ ]:
t = V2.comprobar_trayport(MATRIZ)
print("vetadas               :", t["vetadas"])
print("presentes en la matriz:", t["presentes"] or "ninguna: la matriz ya llega sin Trayport")
for v in VETOS:
    cols = V2.columnas_vetadas(MATRIZ, (v,))
    print(f"veto de frontera {v}: {len(cols)} columnas ·", cols)

## 1 · Entrenar

Lo normal es lanzarlo por terminal, en WSL: ahí están la GPU para las redes y statsmodels y pmdarima para SARIMA. La primera línea es imprescindible: sin ella TensorFlow no encuentra las librerías CUDA de los paquetes `nvidia-*`, dice «Skipping registering GPU devices» y entrena en CPU.

```bash
export LD_LIBRARY_PATH=$(python -c "import glob,site; print(':'.join(glob.glob(site.getsitepackages()[0] + '/nvidia/*/lib')))"):/usr/lib/wsl/lib
python scripts/entrenar_finales_v2.py --matriz nucleo --semillas 3 --guardar-modelos
python scripts/entrenar_finales_v2.py --ancla semanal              # ancla descartada, para comparar
python scripts/entrenar_finales_v2.py --vetar met_lags ree_prev    # ablación de las previsiones REE
```

Cada configuración distinta de la de por defecto escribe en su propia carpeta (`finales_v2_nucleo_anclasemanal`, `finales_v2_nucleo_veto-met_lags+ree_prev`), así que el control y la ablación no se mezclan con el entrenamiento principal. Para leerlas aquí basta con cambiar `ANCLA` o `VETOS` arriba.

Cada entrenamiento escribe en `por_semilla.csv` y al relanzar se salta lo hecho, así que se puede parar y continuar, o lanzar por partes con `--familias`. Lo caro son las redes (en torno a hora y media con GPU) y SARIMA/SARIMAX: el ajuste sobre cinco años de horas y el recorrido día a día de 2025 y 2026. Los órdenes de SARIMA se reutilizan de la búsqueda de Samuel; `--rebuscar-orden` los recalcula con `auto_arima`, y eso son horas.

`--prueba` hace una pasada de humo en minutos (una semilla, dos épocas, 60 árboles, SARIMA sobre cinco días) en la carpeta `*_prueba`.

Desde aquí, con `ENTRENAR = True`:

In [ ]:
ENTRENAR = False
FAMILIAS = V2.FAMILIAS            # o una parte, p. ej. ["lgbm_nucleo", "ridge", "elasticnet"]

if ENTRENAR:
    V2.ejecutar(MATRIZ, semillas=3, familias=FAMILIAS, guardar_modelos=True, prueba=PRUEBA,
                ancla=ANCLA, vetos=VETOS)
else:
    print("ENTRENAR = False: se leen los resultados que haya en", SALIDA.relative_to(REPO))

## 2 · Resultados por familia

Media y desviación sobre las semillas; las familias deterministas no tienen desviación. Ordenado por MAE de validación, que es con lo que se decide: el test se lee, no se usa para elegir.

In [ ]:
R = V2.cargar_resultados(MATRIZ, PRUEBA, ANCLA, VETOS)
if R is None:
    print("Todavía no hay resultados: ejecuta la sección 1.")
else:
    por_semilla, resumen, meta = R["por_semilla"], R["resumen"], R["meta"]
    print(f"matriz {meta['matriz']} · hash {meta['hash']} · naive val {meta['naive_val_MAE']:.2f} "
          f"· test {meta['naive_test_MAE']:.2f} · {meta['dias_train']}/{meta['dias_val']}/{meta['dias_test']} días")
    display(resumen)

## 3 · La copia del día D

**El problema.** Las curvas predichas para D+1 se parecían demasiado a la de D. No era una fuga: 7 de las 8 redes predicen el **residuo** frente al precio de D, y lo que no saben explicar lo dejan en cero, es decir, en D. En el ensemble de 06, sobre test, el perfil horario predicho correlaba 0,948 con el de D cuando el real solo correla 0,858, y la predicción se apartaba de D un 77 % de lo que se aparta la realidad. Los lunes, con D en domingo, eran el peor día.

**Cómo se mide aquí**, por modelo y en test:

- `movimiento`: |pred − D| medio entre |real − D| medio. 0 es copiar D; 100, moverse tanto como la realidad. Quedarse algo por debajo es lo esperable con incertidumbre: amplificar el movimiento del ensemble de 06 subía su MAE de 12,13 a 12,27.
- `forma_vs_D`: correlación media del perfil horario de cada día, sin su nivel, contra el de D. Lo que delata la copia es quedar claramente por encima del valor de la realidad, que se imprime antes de la tabla.

**Qué se probó y se descartó.** Tomar el residuo contra un **ancla semanal**: D corregido con D−6, que cae en el mismo día de la semana que D+1, con un peso por día de la semana y hora ajustado solo con entrenamiento. Por sí sola, como referencia, baja el MAE del naive de 19,95 a 18,84 en validación y de 16,72 a 16,32 en test, y en las redes reduce algo la copia; pero con las trece familias no mejora la validación, que es el criterio: el ensemble seleccionado queda en 11,83 con ella frente a 11,73 con el ancla D. Sigue disponible con `--ancla semanal`, y la última tabla compara las dos carpetas.

Dentro del ensemble final, la copia la compensan los dos miembros de objetivo absoluto, el `lgbm_nucleo` y el `seq2seq_absoluto`, que son los que menos se parecen a la curva de D.

In [ ]:
if R is not None:
    print(f"realidad: perfil de D+1 contra el de D = {meta['forma_real_vs_D_test']:.3f} en test")
    a = meta["ancla"]
    print(f"ancla {a['tipo']}: MAE val {a['MAE_val']:.2f} · test {a['MAE_test']:.2f}")
    if a["pesos_Dm6"]:
        pesos = pd.DataFrame(a["pesos_Dm6"], index=["lun", "mar", "mié", "jue", "vie", "sáb", "dom"])
        print("peso de D-6 según el día de la semana de D+1 (media de las 24 horas):")
        display(pesos.mean(axis=1).round(2).to_frame("peso D-6").T)
    display(resumen[["tipo", "MAE_test", "movimiento", "forma_vs_D"]]
            .sort_values("forma_vs_D", ascending=False))

In [ ]:
import matplotlib.pyplot as plt

DIAS = {"Monday": "lun", "Tuesday": "mar", "Wednesday": "mié", "Thursday": "jue",
        "Friday": "vie", "Saturday": "sáb", "Sunday": "dom"}

if R is not None:
    def leer(nombre):
        return pd.read_csv(SALIDA / nombre, index_col=0, parse_dates=True)

    real, D, ancla = leer("ref_test_real.csv"), leer("ref_test_naive.csv"), leer("ref_test_ancla.csv")
    nombre = next((n for n in ("ensemble_seleccion", "ensemble_todos", "ensemble")
                   if (SALIDA / f"pred_test_{n}.csv").exists()), None)
    if nombre is None:
        print("todavía no hay ensemble")
    else:
        pred = leer(f"pred_test_{nombre}.csv").reindex(real.index)
        dia = real.index.day_name().map(DIAS)
        t = pd.DataFrame({"MAE naive D": (D - real).abs().mean(axis=1),
                          "MAE ancla": (ancla - real).abs().mean(axis=1),
                          f"MAE {nombre}": (pred - real).abs().mean(axis=1),
                          "cambio real": (real - D).abs().mean(axis=1),
                          "cambio predicho": (pred - D).abs().mean(axis=1)}).groupby(dia).mean()
        t["movimiento %"] = 100 * t["cambio predicho"] / t["cambio real"]
        print("por día de la semana de D+1, en test:")
        display(t.reindex(list(DIAS.values())).dropna(how="all").round(1))

        # el día de test con más cambio real frente a D: ¿la predicción se mueve o se queda en D?
        dmax = (real - D).abs().mean(axis=1).idxmax()
        fig, ax = plt.subplots(figsize=(9, 4))
        horas = range(24)
        ax.plot(horas, D.loc[dmax], color="#9A9A9A", lw=2, label="D (naive)")
        if meta["ancla"]["tipo"] != "D":           # con ancla D coincide con el naive
            ax.plot(horas, ancla.loc[dmax], color="#B8650F", lw=1.5, ls="--", label="ancla semanal")
        ax.plot(horas, real.loc[dmax], color="#222222", lw=2, label="real D+1")
        ax.plot(horas, pred.loc[dmax], color="#2F6FC4", lw=2, label=nombre)
        ax.set_title(f"D+1 = {dmax:%Y-%m-%d} ({DIAS[dmax.day_name()]}): el día de test con más cambio frente a D")
        ax.set_xlabel("hora")
        ax.set_ylabel("EUR/MWh")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
if R is not None:
    otra = "D" if ANCLA == "semanal" else "semanal"
    C = V2.cargar_resultados(MATRIZ, PRUEBA, otra, VETOS)
    if C is None:
        print(f"sin control con --ancla {otra}: no existe",
              V2.carpeta_salida(MATRIZ, PRUEBA, otra, VETOS).relative_to(REPO))
    else:
        cols = ["MAE_val", "MAE_test", "movimiento", "forma_vs_D"]
        comun = resumen.index.intersection(C["resumen"].index)
        display(resumen.loc[comun].reindex(columns=cols)
                .join(C["resumen"].loc[comun].reindex(columns=cols),
                      lsuffix=f" · ancla {ANCLA}", rsuffix=f" · ancla {otra}"))

## 4 · La frontera en producción

A las 11:00 del día D el modelo solo tendrá **previsiones** de D+1 y datos ya publicados. Si se entrena con algo que a esa hora no existe, el error medido sale mejor que el que tendrá en producción.

| grupo | qué es | a las 11:00 de D | aquí |
|---|---|---|---|
| `_D` | precio y programas del día D | publicado a las 13:00–13:45 de D−1 | dentro |
| `_Dm1`, `_Dm2`, `_Dm6` | datos reales ya ocurridos | publicados | dentro |
| `*_meteo` | previsión ECMWF de D+1; antes de abril de 2024, pseudo-previsión con el error medido | publicada | dentro |
| `*_met_Dm1`, `*_met_Dm2` | reanálisis ERA5 de D−1 y D−2 | **no existe**: ERA5 va unos 5 días por detrás | **vetado** (`met_lags`) |
| `ree_*_prev` | previsión de REE para D+1 | publicada, pero el histórico se cargó ya **revisado** | dentro; ablación con `--vetar met_lags ree_prev` |

Los `met_lags` solo los usaban los modelos planos, los árboles de Willy y los lineales y el SARIMAX de Samuel: el tensor de las redes no los lleva. Cuando la matriz se reconstruya con `ERA5_PREFERIR_ECMWF` (la previsión en lugar del reanálisis) se pueden devolver con `--vetar` sin argumentos.

Dos comprobaciones quedan fuera del script porque necesitan la base de datos o la API de ESIOS: `scripts/auditoria_frontera.py` mide contra las tablas fuente qué día describe cada columna, e `ingesta/check_tables/verificar_revision_indicadores.py` compara la previsión guardada con la que devuelve hoy la API.

In [ ]:
if R is not None:
    for v, cols in meta.get("vetos_frontera", {}).items():
        print(f"veto {v}: {len(cols)} columnas · {cols}")
    print()
    print("columnas por día que describen, según su nombre:")
    display(pd.DataFrame(meta["frontera"]).fillna(0).astype(int))
    print("previsiones REE en la matriz plana:", meta.get("frontera_ree_prev"))
    for fam in ("lgbm_nucleo", "ridge", "sarimax"):
        ex = V2.leer_extras(SALIDA, fam)
        if ex:
            reanalisis = [c for c in ex["features"] if V2.VETOS["met_lags"](c)]
            print(f"reanálisis ERA5 entre las features de {fam}: {reanalisis or 'ninguno'}")

## 5 · Árboles de Willy · `lgbm_nucleo`

**Por qué.** Es el camino de árboles que Willy desarrolló por su cuenta, y aporta dos diferencias frente a `boosting`: **un solo modelo para las 24 horas**, que comparte lo aprendido entre horas en lugar de entrenar 24 por separado, y la **matriz fila a fila** en lugar de la vista resumida del encoder.

**Cómo.** `LGBMRegressor` sobre las filas horarias de entrenamiento, con el precio absoluto como objetivo. Columnas: todas las de la matriz salvo control, banderas de trazabilidad, las siete de baterías con arranque tardío, Trayport y los retardos de reanálisis ERA5 (sección 4). Hiperparámetros: los ganadores de su campaña de Optuna (300 pruebas sobre su matriz horaria), sin reafinar sobre núcleo, como hizo él. A diferencia de su versión, con tres semillas: con submuestreo de filas (0,60) y de columnas (0,77), la semilla sí cambia el modelo.

Referencia: su reentrenamiento sin Trayport en la rama `willy_test` dio 13,24 €/MWh de MAE en test con 120 features (nota 54).

In [ ]:
if R is not None:
    print("hiperparámetros:", V2.LGBM_WILLY)
    display(por_semilla[por_semilla.familia == "lgbm_nucleo"])
    ex = V2.leer_extras(SALIDA, "lgbm_nucleo")
    if ex:
        con_trayport = sorted(set(ex["features"]) & set(V2.TRAYPORT))
        print(f"{len(ex['features'])} features · Trayport entre ellas: {con_trayport or 'ninguna'}")
    imp = V2.importancias_lgbm(SALIDA)
    if imp is None:
        print("sin modelos guardados: relanza con --guardar-modelos para ver las importancias")
    else:
        display(imp.head(15))

## 6 · Clásicos de Samuel · `ridge`, `elasticnet`, `sarima`, `sarimax`

**Por qué.** Son la referencia estadística del temario: si una red o un árbol no batiera a una regresión penalizada o a un SARIMA, su complejidad no se justificaría.

**Selección de variables.** Su filtro de Spearman, con la misma función y los mismos umbrales —|ρ| ≥ 0,10 con el precio y p < 0,05, y de cada par con |ρ| ≥ 0,85 se queda la más correlada con el precio—, ajustado solo con entrenamiento. Las variables de hora y calendario están protegidas: su relación con el precio es en U y Spearman las descartaría.

**Ridge y ElasticNet.** Rejilla de `alpha` (y de `l1_ratio` en ElasticNet) elegida por MAE de validación sobre variables tipificadas; el modelo final lleva el `StandardScaler` dentro del pipeline. Como el hiperparámetro se elige mirando validación, esa métrica queda algo optimista.

**SARIMA y SARIMAX.** Estacionalidad diaria (m = 24) y los órdenes de su `auto_arima`: SARIMA(3,1,0)(1,0,0)[24] y, para SARIMAX, (0,0,0)(0,0,0)[24], sin parte ARIMA: con un centenar de exógenas, `auto_arima` prefirió una regresión pura. Ajuste sobre 2020–2024 con `low_memory` y predicción por bloques de un día: cada día objetivo se predice entero y después se le da al modelo el día observado, que es lo que se sabe a las 11:00. Las exógenas de SARIMAX entran tipificadas con la media y la desviación de la ventana de ajuste, sin las constantes y con un término independiente: el orden (0,0,0)(0,0,0) es una regresión sin intercepto, y con exógenas de media cero no puede representar el nivel del precio (sin él, el ajuste completo predecía con un sesgo de −80 €/MWh). La rejilla de Ridge se amplía por arriba, porque el óptimo caía en su borde.

Referencia: en la clasificación común de validación 2025 (`data_temp/leaderboard_validation.csv`) marcaban Ridge 14,84, ElasticNet 14,84, SARIMA 21,28 y SARIMAX 36,43 €/MWh, con el naive en 19,99.

In [ ]:
if R is not None:
    clasicos = ["ridge", "elasticnet", "sarima", "sarimax"]
    display(por_semilla[por_semilla.familia.isin(clasicos)])
    for fam in clasicos:
        ex = V2.leer_extras(SALIDA, fam)
        if ex:
            detalle = ex.get("hiperparametros") or {"orden": ex.get("orden")}
            print(f"{fam:11s} {len(ex.get('features', []))} features · {detalle}")
    for fam in ("ridge", "elasticnet"):
        rejilla = SALIDA / f"tuning_{fam}.csv"
        if rejilla.exists():
            print(f"\nrejilla de {fam}, por MAE de validación:")
            display(pd.read_csv(rejilla).head(8))

## 7 · Ensembles

Tres, todos decididos solo con validación:

- `ensemble_seleccion`, **el modelo final**: selección voraz de Caruana et al. (2004). En cada paso se añade, con reemplazo, el representante de familia que más baja el MAE de validación de la media, y se queda el tamaño con menor error. Un modelo solo entra si mejora al conjunto, y las veces que se elige son su peso. Su MAE de validación es optimista, porque se ha optimizado sobre él; el de test es la medida honesta.
- `ensemble`: la regla de 06 (un representante por familia que bata al naive en validación), solo con las 8 familias de 06.
- `ensemble_todos`: la misma regla con todas las familias. Deja entrar a modelos que baten al naive en validación pero no en test, como `ridge` y `sarimax`, y por eso se sustituyó por la selección.

In [ ]:
if R is not None:
    for nombre, miembros in meta.get("ensembles", {}).items():
        if isinstance(miembros, dict):             # la seleccion guarda el peso de cada miembro
            detalle = ", ".join(f"{k} {w:.0%}" for k, w in miembros.items())
        else:
            detalle = ", ".join(miembros)
        print(f"{nombre}: {detalle}")
    display(resumen.loc[[i for i in ("ensemble_seleccion", "ensemble", "ensemble_todos", "ancla semanal",
                                     "naive (persistencia)") if i in resumen.index]])

## 8 · MAE contra captura

Como en 06: el campeón por error y el campeón por captura no tienen por qué coincidir, y para operar una batería importa acertar **cuándo** cargar y descargar.

In [ ]:
if R is not None:
    r = resumen[~resumen.tipo.isin(["referencia"])].dropna(subset=["MAE_test", "captura_test"])
    colores = {"red": "#2F6FC4", "arbol": "#2E8B57", "lineal": "#B8650F",
               "estadistico": "#8E44AD", "ensemble": "#222222"}
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for fam, fila in r.iterrows():
        ax.scatter(fila.MAE_test, fila.captura_test, s=60, color=colores.get(fila.tipo, "#777777"))
        ax.annotate(fam, (fila.MAE_test, fila.captura_test), textcoords="offset points",
                    xytext=(5, 4), fontsize=8)
    ax.set_xlabel("MAE test (EUR/MWh)")
    ax.set_ylabel("captura test (%)")
    ax.set_title("Precisión frente a captura, todas las familias")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    individuales = r[r.tipo != "ensemble"]
    print("mejor MAE     :", individuales.MAE_test.idxmin())
    print("mejor captura :", individuales.captura_test.idxmax())

## 9 · Qué queda guardado

`por_semilla.csv` y `resumen.csv` con las métricas (también las de copia), `pred_val_*` y `pred_test_*` por familia y semilla, `ref_{val,test}_{real,naive,ancla}.csv` con las referencias de la sección 3, `extras_*.json` con las features, hiperparámetros, órdenes y tipificación de cada familia nueva, `tuning_*.csv` con las rejillas de los lineales y `meta.json` con la matriz, los vetos de Trayport y de frontera, el ancla con sus pesos y los miembros de cada ensemble. Con `--guardar-modelos`, además, cada artefacto con su `.preprocesado.json`.

In [ ]:
if SALIDA.exists():
    for f in sorted(SALIDA.iterdir()):
        tam = sum(p.stat().st_size for p in f.rglob("*")) if f.is_dir() else f.stat().st_size
        print(f"{tam / 1024:10,.0f} KB  {f.name}")
else:
    print("todavía no existe", SALIDA.relative_to(REPO))